In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS databricks_worksoace.default;

CREATE OR REPLACE TABLE databricks_worksoace.default.calender
USING DELTA
AS
WITH base_dates AS (
  SELECT explode(sequence(to_date('1940-01-01'), to_date('2030-12-31'), interval 1 day)) AS calendar_date
)
SELECT
  CAST(date_format(calendar_date, 'yyyyMMdd') AS INT) AS date_key,
  calendar_date AS `date`,
  year(calendar_date) AS year,
  quarter(calendar_date) AS quarter,
  concat('Q', quarter(calendar_date)) AS quarter_name,
  concat(year(calendar_date), '-Q', quarter(calendar_date)) AS year_quarter,
  CASE WHEN month(calendar_date) BETWEEN 1 AND 6 THEN 1 ELSE 2 END AS half_year,
  month(calendar_date) AS month,
  date_format(calendar_date, 'MMMM') AS month_name,
  date_format(calendar_date, 'MMM') AS month_short_name,
  CAST(date_format(calendar_date, 'yyyyMM') AS INT) AS year_month_key,
  date_format(calendar_date, 'yyyy-MM') AS year_month,
  day(calendar_date) AS day_of_month,
  dayofweek(calendar_date) AS day_of_week,
  pmod(dayofweek(calendar_date) + 5, 7) + 1 AS iso_day_of_week,
  date_format(calendar_date, 'EEEE') AS day_name,
  date_format(calendar_date, 'E') AS day_short_name,
  dayofyear(calendar_date) AS day_of_year,
  weekofyear(calendar_date) AS week_of_year,
  format_string('%04d-W%02d', year(calendar_date), weekofyear(calendar_date)) AS year_week,
  date_sub(calendar_date, pmod(dayofweek(calendar_date) + 5, 7)) AS week_start_date,
  date_add(date_sub(calendar_date, pmod(dayofweek(calendar_date) + 5, 7)), 6) AS week_end_date,
  trunc(calendar_date, 'month') AS month_start_date,
  last_day(calendar_date) AS month_end_date,
  CAST(date_trunc('quarter', calendar_date) AS DATE) AS quarter_start_date,
  date_sub(add_months(CAST(date_trunc('quarter', calendar_date) AS DATE), 3), 1) AS quarter_end_date,
  CAST(date_trunc('year', calendar_date) AS DATE) AS year_start_date,
  date_sub(add_months(CAST(date_trunc('year', calendar_date) AS DATE), 12), 1) AS year_end_date,
  day(calendar_date) = 1 AS is_month_start,
  calendar_date = last_day(calendar_date) AS is_month_end,
  calendar_date = CAST(date_trunc('quarter', calendar_date) AS DATE) AS is_quarter_start,
  calendar_date = date_sub(add_months(CAST(date_trunc('quarter', calendar_date) AS DATE), 3), 1) AS is_quarter_end,
  calendar_date = CAST(date_trunc('year', calendar_date) AS DATE) AS is_year_start,
  calendar_date = date_sub(add_months(CAST(date_trunc('year', calendar_date) AS DATE), 12), 1) AS is_year_end,
  dayofweek(calendar_date) IN (1, 7) AS is_weekend
FROM base_dates;

In [0]:
%sql
SELECT
  COUNT(*) AS row_count,
  MIN(`date`) AS min_date,
  MAX(`date`) AS max_date,
  MIN(date_key) AS min_date_key,
  MAX(date_key) AS max_date_key
FROM databricks_worksoace.default.calender;

In [0]:
%sql
SELECT *
FROM databricks_worksoace.default.calender
ORDER BY `date`
LIMIT 5;